# 03. Feature Engineering & Target Labeling

Create the **valuation_label** target via peer-group z-score, plus the engineered feature set.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np


In [ ]:
from src.utils.config import load_config
from src.utils.io import load_parquet
from src.data.enrichment import PropertyEnricher
from src.features.engineering import FeatureEngineer

cfg = load_config()
df = load_parquet(f"{cfg['paths']['data_raw']}/properties.parquet")
print('Raw:', df.shape)

## Add neighborhood-percentile features

In [ ]:
enricher = PropertyEnricher(cfg)
enriched = enricher.enrich_all(df)
print('Enriched:', enriched.shape)
enriched.head()

## Build engineered features + create labels

In [ ]:
fe = FeatureEngineer(cfg)
features = fe.run(enriched)
print('Final:', features.shape)
print()
print(features['valuation_label_name'].value_counts())

## Visualize label distribution

In [ ]:
from src.visualization.eda_plots import plot_label_distribution
plot_label_distribution(features, 'reports/figures/label_distribution.png')

## Inspect peer-z distribution

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.hist(features['peer_z'].clip(-3,3), bins=80, color='#1f77b4')
plt.axvline(-0.75, color='r', ls='--')
plt.axvline(0.75, color='r', ls='--')
plt.title('Peer-group z-score distribution\n(red lines = label thresholds)')
plt.xlabel('z-score'); plt.ylabel('count')
plt.tight_layout()
plt.savefig('reports/figures/peer_z_dist.png', dpi=110)
plt.show()

## Save

In [ ]:
from src.utils.io import save_parquet
save_parquet(features, f"{cfg['paths']['data_processed']}/properties_features.parquet")